In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from tqdm import tqdm

In [2]:
DATA_FOLDER = Path("../data")
OUTPUT_FOLDER = Path("../outputs")

In [3]:
def fetch_candidates(df, file: str, pattern: str = r"^\d{3}-") -> pd.DataFrame:
    # Collect all new dataframes in a fast Python list
    new_dfs = []

    for subfolder in tqdm(OUTPUT_FOLDER.iterdir()):
        if not re.search(pattern, subfolder.name):
            print(f"Skipping subfolder {subfolder.name} as it does not match the pattern.")
            continue
        json_file = subfolder / file
        if not json_file.exists():
            print(f"Skipping subfolder {subfolder.name} as {file} does not exist.")
            continue
        with open(json_file, "r") as f:
            data = json.load(f)

        temp_df = pd.DataFrame(data)
        temp_df["run_name"] = subfolder.name
        new_dfs.append(temp_df)

    if new_dfs:
        return pd.concat([df] + new_dfs, ignore_index=True)

    return df


### Test Individual Runs

In [10]:
import pandas as pd
import plotly.figure_factory as ff

test = pd.DataFrame()
# test = fetch_candidates(
#     test,
#     file="candidate_passages_before_threshold.json",
#     pattern=r"^[^\d]*$"
# )
test = fetch_candidates(
    test,
    file="candidate_passages_before_threshold.json",
    pattern=r"\d+"
)

plot_data = test.dropna(
    subset=["rerank_score", "topic", "retrieval_query"]
)

groups = [
    group["rerank_score"].to_numpy()
    for _, group in plot_data.groupby("retrieval_query")
]

labels = [
    str(label)
    for label, _ in plot_data.groupby("retrieval_query")
]

fig = ff.create_distplot(
    hist_data=groups,
    group_labels=labels,
    show_hist=False,
    show_rug=False,
    curve_type="kde"
)

fig.update_layout(
    xaxis_title="Rerank Score",
    yaxis_title="Density",
    legend_title_text="Retrieval Query",
    template="plotly_white"
)

fig.add_vline(
    x=0.40,
    line_dash="dash",
    line_width=1,
    annotation_text="Threshold = 0.40"
)

fig.show()

0it [00:00, ?it/s]

47it [00:00, 234.74it/s]

Skipping subfolder free-trade-us-deepseek as it does not match the pattern.
Skipping subfolder free-trade-us-demo_deepseek as it does not match the pattern.
Skipping subfolder free-trade-us-qwen as it does not match the pattern.
Skipping subfolder annotation_subset_sophia.csv as it does not match the pattern.
Skipping subfolder free-trade-us-demo_qwen as it does not match the pattern.
Skipping subfolder free-trade-us-demo_mdeberta as it does not match the pattern.
Skipping subfolder annotation_subset_laurence.csv as it does not match the pattern.
Skipping subfolder annotation_candidates_above_0.40.json as candidate_passages_before_threshold.json does not exist.
